# Module 2 — Proximity Enrichment

**Input:** latest `osm_commercial_*.csv` from `osm_commercial_fetch.ipynb`  
**Output:** `osm_distances_*.csv` with 8 new columns:

| Column | Description |
|---|---|
| `dist_transit_m` | Meters to nearest transit stop |
| `nearest_transit` | Name of that stop |
| `dist_hospital_m` | Meters to nearest hospital/clinic |
| `nearest_hospital` | Name of that hospital |
| `dist_school_m` | Meters to nearest school/university |
| `nearest_school` | Name of that school |
| `dist_park_m` | Meters to nearest park/garden |
| `nearest_park` | Name of that park |

---
- Run cells **top to bottom**
- **No parameters to edit** — origin and radius are inferred from the input CSV
- Edit `PROXIMITY_TAGS` in Cell 1 to change which OSM features count as transit/hospital/etc.
- Kernel: **OSMnx Scraper (Python 3.11)**

## Cell 1 · Imports & Proximity Tag Config

In [8]:
import overpy, requests, pandas as pd, math, time, glob
from datetime import datetime

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

# Proximity categories — edit here if you need different OSM tags
PROXIMITY_TAGS = {
    "transit": [
        ("highway",          "bus_stop"),
        ("railway",          "subway_entrance"),
        ("railway",          "station"),
        ("railway",          "tram_stop"),
        ("public_transport", "stop_position"),
        ("amenity",          "bus_station"),
        ("amenity",          "ferry_terminal"),
    ],
    "hospital": [
        ("amenity", "hospital"),
        ("amenity", "clinic"),
    ],
    "school": [
        ("amenity", "school"),
        ("amenity", "university"),
        ("amenity", "college"),
    ],
    "park": [
        ("leisure", "park"),
        ("leisure", "garden"),
        ("landuse", "recreation_ground"),
    ],
}

print(f"overpy {overpy.__version__} · pandas {pd.__version__} · Ready")


overpy 0.7 · pandas 3.0.2 · Ready


## Cell 2 · Load Input CSV

Auto-loads the most recent `osm_commercial_*.csv`.  
Origin coordinates and search radius are inferred automatically from the data.

In [9]:
# ── Reads the most recent CSV from osm_commercial_fetch.ipynb ─────────────────
# papermill tag: parameters
files = sorted(glob.glob("osm_commercial_*.csv"))
if not files:
    raise FileNotFoundError(
        "No osm_commercial_*.csv found. "
        "Run osm_commercial_fetch.ipynb first."
    )

INPUT_CSV = files[-1]
df = pd.read_csv(INPUT_CSV)

# Infer the scrape origin from the bounding box center of the data
# (close enough to drive the proximity search radius)
ORIGIN_LAT = (df["lat"].max() + df["lat"].min()) / 2
ORIGIN_LON = (df["lon"].max() + df["lon"].min()) / 2

# Proximity search radius = max feature distance + 1500 m buffer
PROXIMITY_RADIUS_M = df["distance_m"].max() + 1500

# Output file
_existing   = glob.glob("osm_distances_*.csv")
OUTPUT_PATH = f"osm_distances_{len(_existing)+1:03d}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"

print(f"Input    : {INPUT_CSV}  ({len(df)} rows)")
print(f"Origin   : ({ORIGIN_LAT:.5f}, {ORIGIN_LON:.5f})  [inferred from data]")
print(f"Search R : {PROXIMITY_RADIUS_M:.0f} m")
print(f"Output   : {OUTPUT_PATH}")


Input    : osm_commercial_001_2026-04-30_17-28-10.csv  (3704 rows)
Origin   : (40.76534, -73.98230)  [inferred from data]
Search R : 4602 m
Output   : osm_distances_001_2026-04-30_17-28-43.csv


## Cell 3 · Functions — do not edit

In [10]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl  = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def overpass_query(query):
    last = None
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            return overpy.Overpass().parse_json(r.text)
        except Exception as e:
            last = e
            print(f"  x {ep}: {type(e).__name__}: {e}")
            time.sleep(3)
    raise RuntimeError(f"All Overpass endpoints failed. Last: {last}")


def coords_of(el):
    if hasattr(el, "lat"):
        return float(el.lat), float(el.lon)
    if hasattr(el, "center_lat") and el.center_lat:
        return float(el.center_lat), float(el.center_lon)
    return None


def fetch_proximity_pois(lat, lon, radius):
    # Fetches transit / hospital / school / park POIs in one Overpass call.
    tag_to_cat = {}
    lines = []
    for cat, pairs in PROXIMITY_TAGS.items():
        for key, val in pairs:
            tag_to_cat[(key, val)] = cat
            lines.append(f'node["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')
            lines.append(f'way["{key}"="{val}"](around:{radius:.0f},{lat},{lon});')

    query = (
        "[out:json][timeout:90];\n(\n    "
        + "\n    ".join(lines)
        + "\n);\nout center tags;"
    )

    print(f"Fetching proximity POIs (radius {radius:.0f} m)...")
    result = overpass_query(query)

    pois = {cat: [] for cat in PROXIMITY_TAGS}
    for el in list(result.nodes) + list(result.ways):
        c = coords_of(el)
        if not c:
            continue
        elat, elon = c
        name = el.tags.get("name", "")
        for (key, val), cat in tag_to_cat.items():
            if el.tags.get(key) == val:
                pois[cat].append((elat, elon, name))
                break

    for cat, items in pois.items():
        print(f"  {cat:10s}: {len(items)} POIs")
    return pois


def add_distances(df, pois):
    # Adds 8 columns: dist_transit_m / nearest_transit, dist_hospital_m / nearest_hospital,
    # dist_school_m / nearest_school, dist_park_m / nearest_park
    for cat, poi_list in pois.items():
        dists, names = [], []
        for _, row in df.iterrows():
            if not poi_list:
                dists.append(None)
                names.append("")
                continue
            best_d, best_n = math.inf, ""
            for plat, plon, pname in poi_list:
                d = haversine(row["lat"], row["lon"], plat, plon)
                if d < best_d:
                    best_d, best_n = d, pname
            dists.append(round(best_d, 1))
            names.append(best_n)
        df[f"dist_{cat}_m"]  = dists
        df[f"nearest_{cat}"] = names
        filled = [d for d in dists if d is not None]
        if filled:
            print(f"  dist_{cat}_m: min {min(filled):.0f} m  avg {sum(filled)/len(filled):.0f} m  max {max(filled):.0f} m")
        else:
            print(f"  WARNING: no {cat} POIs found — increase PROXIMITY_RADIUS_M")
    return df


print("Functions ready.")


Functions ready.


## Cell 4 · Run

Queries Overpass for proximity POIs and computes nearest distances for every feature.  
Expect **20–60 seconds**.

In [11]:
# ── Fetch POIs ────────────────────────────────────────────────────────────────
pois = fetch_proximity_pois(ORIGIN_LAT, ORIGIN_LON, PROXIMITY_RADIUS_M)

# ── Compute nearest distances ─────────────────────────────────────────────────
print("\nComputing nearest distances...")
df = add_distances(df, pois)

# ── Save ──────────────────────────────────────────────────────────────────────
df.sort_values("distance_m", inplace=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"\n>>> SAVED: {OUTPUT_PATH}")
print(f"    Rows   : {len(df)}")
print(f"    Columns: {len(df.columns)}")
print(f"    List   : {list(df.columns)}")


Fetching proximity POIs (radius 4602 m)...
  transit   : 2864 POIs
  hospital  : 186 POIs
  school    : 358 POIs
  park      : 1993 POIs

Computing nearest distances...
  dist_transit_m: min 0 m  avg 62 m  max 282 m
  dist_hospital_m: min 0 m  avg 242 m  max 772 m
  dist_school_m: min 0 m  avg 213 m  max 608 m
  dist_park_m: min 0 m  avg 122 m  max 389 m

>>> SAVED: osm_distances_001_2026-04-30_17-28-43.csv
    Rows   : 3704
    Columns: 37
    List   : ['osm_id', 'osm_type', 'lat', 'lon', 'distance_m', 'primary_tag', 'primary_value', 'name', 'brand', 'shop', 'amenity', 'office', 'craft', 'tourism', 'leisure', 'cuisine', 'opening_hours', 'phone', 'website', 'wheelchair', 'outdoor_seating', 'takeaway', 'delivery', 'level', 'addr_street', 'addr_housenumber', 'addr_postcode', 'has_name', 'distance_band', 'dist_transit_m', 'nearest_transit', 'dist_hospital_m', 'nearest_hospital', 'dist_school_m', 'nearest_school', 'dist_park_m', 'nearest_park']


## Cell 5 · Preview

In [12]:
df[[
    "name", "primary_value", "distance_m",
    "dist_transit_m", "nearest_transit",
    "dist_hospital_m", "nearest_hospital",
    "dist_school_m", "nearest_school",
    "dist_park_m", "nearest_park",
]].head(15)


,name,primary_value,distance_m,dist_transit_m,nearest_transit,dist_hospital_m,nearest_hospital,dist_school_m,nearest_school,dist_park_m,nearest_park
0,NaN,outdoor_seating,13.5,62.6,,421.5,Manhattan Gastroenterology – Midtown Manhattan,329.8,Repertory Company High School for Theatre Arts,25.3,
1,NaN,garden,16.1,70.4,,399.2,Manhattan Gastroenterology – Midtown Manhattan,304.6,Repertory Company High School for Theatre Arts,0.0,
2,George M. Cohan,artwork,16.1,70.4,,399.2,Manhattan Gastroenterology – Midtown Manhattan,304.6,Repertory Company High School for Theatre Arts,0.0,
3,Francis P. Duffy,artwork,19.8,47.2,,415.3,Manhattan Gastroenterology – Midtown Manhattan,335.5,Repertory Company High School for Theatre Arts,35.6,
4,NaN,newsagent,24.6,73.2,,428.4,Manhattan Gastroenterology – Midtown Manhattan,329.1,Repertory Company High School for Theatre Arts,29.1,
5,Walk NYC,information,25.6,80.1,,424.0,Manhattan Gastroenterology – Midtown Manhattan,322.1,Repertory Company High School for Theatre Arts,25.5,
6,TKTS,ticket,31.7,35.6,,420.4,Manhattan Gastroenterology – Midtown Manhattan,345.6,Repertory Company High School for Theatre Arts,41.6,
7,NaN,viewpoint,31.7,35.6,,420.4,Manhattan Gastroenterology – Midtown Manhattan,345.6,Repertory Company High School for Theatre Arts,41.6,
8,American Eagle Outfitters,clothes,33.1,86.0,,428.0,Manhattan Gastroenterology – Midtown Manhattan,322.1,Repertory Company High School for Theatre Arts,31.3,
9,Grand Slam New York,gift,34.2,58.5,,442.3,Manhattan Gastroenterology – Midtown Manhattan,332.8,Public School 212,45.0,


In [13]:
prox = ["dist_transit_m", "dist_hospital_m", "dist_school_m", "dist_park_m"]
df[prox].describe().round(1)


,dist_transit_m,dist_hospital_m,dist_school_m,dist_park_m
count,3704.0,3704.0,3704.0,3704.0
mean,61.7,242.3,213.0,121.5
std,42.4,118.6,108.8,80.1
min,0.0,0.0,0.0,0.0
25%,27.7,158.2,130.9,59.7
50%,53.5,227.3,192.6,113.8
75%,88.3,319.8,285.0,175.4
max,282.0,771.7,607.9,389.0
